# EDA и эксперименты

Этот ноутбук фиксирует разведочный анализ и экспериментальный протокол для проекта по оценке сложности английского текста. Основной исполняемый код находится в `src/text_complexity`, а ноутбук используется как читаемый журнал экспериментов.

In [ ]:
import csv
from collections import Counter
from pathlib import Path


def resolve_existing_path(*candidates: str) -> Path:
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return path
    return Path(candidates[-1])


data_path = resolve_existing_path(
    'project/data/cefr_long_en.csv',
    '../data/cefr_long_en.csv',
    'data/cefr_long_en.csv',
)

with data_path.open(encoding='utf-8', newline='') as file:
    rows = list(csv.DictReader(file))

len(rows), rows[0]

In [ ]:
targets = [float(row['target']) for row in rows]
levels = Counter(row['cefr_level'] for row in rows)
word_lengths = [len(row['text'].split()) for row in rows]

{
    'rows': len(rows),
    'target_min': min(targets),
    'target_max': max(targets),
    'target_mean': round(sum(targets) / len(targets), 4),
    'levels': dict(levels),
    'min_words': min(word_lengths),
    'max_words': max(word_lengths),
}

## График распределения уровней CEFR

Следующая ячейка строит столбчатую диаграмму распределения текстов по уровням CEFR и сохраняет её в `artifacts/cefr_level_distribution.png`.

In [ ]:
import os
artifact_dir = resolve_existing_path(
    'project/artifacts',
    '../artifacts',
    'artifacts',
)
artifact_dir.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str((artifact_dir / '.matplotlib').resolve()))
import matplotlib.pyplot as plt
from matplotlib import font_manager

preferred_fonts = ['Arial', 'Segoe UI', 'Tahoma', 'Verdana', 'Times New Roman', 'DejaVu Sans']
available_fonts = {font.name for font in font_manager.fontManager.ttflist}
selected_font = next((font for font in preferred_fonts if font in available_fonts), 'DejaVu Sans')
plt.rcParams['font.family'] = selected_font
plt.rcParams['axes.unicode_minus'] = False

levels_order = ['A1', 'A2', 'B1', 'B2', 'C1', 'C2']
level_counts = [levels.get(level, 0) for level in levels_order]
colors = ['#4caf50', '#8bc34a', '#ffc107', '#ff9800', '#ff7043', '#e53935']

fig, ax = plt.subplots(figsize=(9, 5), dpi=150)
bars = ax.bar(levels_order, level_counts, color=colors, edgecolor='#333333', linewidth=0.8)
ax.set_title('Распределение текстов по уровням CEFR в датасете cefr_long_en.csv')
ax.set_xlabel('Уровень CEFR')
ax.set_ylabel('Количество текстов')
ax.grid(axis='y', linestyle='--', alpha=0.35)
ax.set_axisbelow(True)

for bar, value in zip(bars, level_counts):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 3, str(value), ha='center', va='bottom')

figure_path = artifact_dir / 'cefr_level_distribution.png'
fig.tight_layout()
fig.savefig(figure_path, bbox_inches='tight')
plt.show()
plt.close(fig)

{'figure_path': str(figure_path), 'font': selected_font}

## Протокол эксперимента

- Базовый ориентир: константное предсказание, равное среднему значению target на обучении.
- Финальная модель: ridge-регрессия по readability-признакам, CEFR-ориентированным лексическим признакам и TF-IDF-подобным признакам по униграммам и биграммам.
- Разбиение: 80/20 при `random_state = 42`.
- Метрики: MAE, RMSE, R2, Spearman, macro F1 и adjacent accuracy.

Запускать из корня проекта:

```bash
python -m src.text_complexity.train --config configs/config.yaml
```

In [ ]:
import json

metrics_path = resolve_existing_path(
    'project/artifacts/metrics.json',
    '../artifacts/metrics.json',
    'artifacts/metrics.json',
)

if metrics_path.exists():
    json.loads(metrics_path.read_text(encoding='utf-8'))
else:
    'Сначала запустите обучение, чтобы появился artifacts/metrics.json'